# ⚡ Segment Anything (SAM) PCB Polygon Point Extractor & Spreadsheet Exporter

This notebook runs Meta's **Segment Anything Model (SAM ViT-B)** on any Printed Circuit Board (PCB) image:
1. Ingests component bounding boxes (`[x1, y1, x2, y2]`)
2. Prompts SAM to extract pixel-exact segmentation masks
3. Simplifies boundaries into clean polygon contour points `[(x, y), ...]` via OpenCV
4. Exports **Excel (`.xlsx`)** & **CSV (`.csv`)** spreadsheets containing all polygon coordinates
5. Generates **LabelMe JSON (`.json`)** with `ref_des: class` tags
6. Displays high-resolution visual previews inline

In [ ]:
# 1. Environment Setup & Dependency Installation
!pip install -q opencv-python numpy pandas openpyxl matplotlib
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import os
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from segment_anything import sam_model_registry, SamPredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Active compute device: {device}')

In [ ]:
# 2. Load SAM ViT-B Checkpoint
import urllib.request

SAM_CHECKPOINT = Path('sam_vit_b.pth')
if not SAM_CHECKPOINT.exists():
    print('Downloading SAM ViT-B weights (375 MB)...')
    urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth', str(SAM_CHECKPOINT))
    print('Download complete!')

print('Loading SAM model...')
sam = sam_model_registry['vit_b'](checkpoint=str(SAM_CHECKPOINT))
sam.to(device=device)
sam.eval()
predictor = SamPredictor(sam)
print('SAM Predictor ready!')

In [ ]:
# 3. Contour Extraction Algorithm (Douglas-Peucker Simplification)
def extract_polygon_points(mask: np.ndarray, min_area: float = 15.0):
    """Converts a binary boolean mask into an ordered list of 2D polygon points [(x, y), ...]."""
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []
    cnt = max(contours, key=cv2.contourArea)
    if cv2.contourArea(cnt) < min_area:
        return []
    epsilon = 0.005 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, epsilon, True)
    return [[round(float(p[0][0]), 2), round(float(p[0][1]), 2)] for p in approx]

In [ ]:
# 4. Run SAM on Target Board & Extract Polygon Points
# Specify your board image and bounding box label file
image_path = Path('dataset_split/train/images/VID20210601143927-96_jpg.rf.36de73b8200ee94d0bd4679407c9cd40.jpg')
labels_path = Path('dataset_split/train/labels/VID20210601143927-96_jpg.rf.36de73b8200ee94d0bd4679407c9cd40.txt')
output_dir = Path('sam_points_output')
output_dir.mkdir(parents=True, exist_ok=True)

img = cv2.imread(str(image_path))
h, w = img.shape[:2]

# Encode PCB image embedding with SAM Vision Transformer
predictor.set_image(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

CLASS_MAP = {0: 'Cap1', 1: 'Cap2', 2: 'Cap3', 3: 'Cap4', 4: 'MOSFET', 5: 'Mov', 6: 'Resistor', 7: 'Transformer'}
PREFIX_MAP = {'Cap1': 'C', 'Cap2': 'C', 'Cap3': 'C', 'Cap4': 'C', 'MOSFET': 'Q', 'Mov': 'D', 'Resistor': 'R', 'Transformer': 'T'}

# Parse bounding boxes
boxes = []
with open(labels_path, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 5:
            cid = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = int(round((xc - bw / 2.0) * w))
            y1 = int(round((yc - bh / 2.0) * h))
            x2 = int(round((xc + bw / 2.0) * w))
            y2 = int(round((yc + bh / 2.0) * h))
            boxes.append(([x1, y1, x2, y2], cid))

records = []
shapes = []
ref_counts = {}
vis_img = img.copy()

print(f'Segmenting {len(boxes)} components with SAM...')
for idx, (b, cid) in enumerate(boxes, 1):
    input_box = np.array(b)
    masks, scores, _ = predictor.predict(box=input_box[None, :], multimask_output=False)
    mask = masks[0]
    conf = float(scores[0])
    pts = extract_polygon_points(mask)
    if not pts:
        pts = [[float(b[0]), float(b[1])], [float(b[2]), float(b[1])], [float(b[2]), float(b[3])], [float(b[0]), float(b[3])]]
    
    class_name = CLASS_MAP.get(cid, f'Class_{cid}')
    prefix = PREFIX_MAP.get(class_name, 'U')
    ref_counts[prefix] = ref_counts.get(prefix, 0) + 1
    ref_des = f'{prefix}{ref_counts[prefix]}'
    
    records.append({
        'image_name': image_path.name,
        'instance_id': idx,
        'ref_des': ref_des,
        'class_name': class_name,
        'box_x1': b[0], 'box_y1': b[1], 'box_x2': b[2], 'box_y2': b[3],
        'confidence': round(conf, 3),
        'num_polygon_points': len(pts),
        'polygon_points_compact': '; '.join([f'({p[0]},{p[1]})' for p in pts]),
        'polygon_points_json': json.dumps(pts)
    })
    
    shapes.append({
        'label': f'{ref_des}: {class_name}',
        'points': pts,
        'shape_type': 'polygon'
    })
    
    cnt = np.array(pts, dtype=np.int32).reshape((-1, 1, 2))
    cv2.polylines(vis_img, [cnt], True, (0, 255, 255), 2)
    cv2.putText(vis_img, ref_des, (b[0], max(20, b[1] - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

# Export Spreadsheets
df = pd.DataFrame(records)
xlsx_path = output_dir / 'sam_output_points.xlsx'
csv_path = output_dir / 'sam_output_points.csv'
df.to_excel(xlsx_path, index=False)
df.to_csv(csv_path, index=False)

# Export LabelMe JSON
json_path = output_dir / f'{image_path.stem}.json'
labelme_payload = {
    'version': '5.5.0',
    'flags': {},
    'shapes': shapes,
    'imagePath': image_path.name,
    'imageData': None,
    'imageHeight': h,
    'imageWidth': w
}
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(labelme_payload, f, indent=2)

print(f'Successfully exported:')
print(f'  - Excel:   {xlsx_path}')
print(f'  - CSV:     {csv_path}')
print(f'  - LabelMe: {json_path}')
df[['ref_des', 'class_name', 'confidence', 'num_polygon_points', 'polygon_points_compact']].head(10)

In [ ]:
# 5. Display Inline Board Visualization with SAM Polygon Overlays
plt.figure(figsize=(12, 10))
plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
plt.title('SAM Polygon Segmentations & KiCad Reference Designators', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()